# Notebook 1: Deidentification of Validation Excel & Source Documents

**Purpose:** Remove PHI from:
1. The validation Excel spreadsheet
2. Source PDF documents (scanned radiology/pathology reports)

**Key requirement:** Maintain a secure `case_id <-> original_filename` mapping table so every deidentified observation can be traced back for downstream observation-level analysis.

**Template:** Based on `deidentify_source_docs.py`

In [ ]:
import os
import re
import uuid
import hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Tuple, Optional

import fitz  # PyMuPDF
import pytesseract
import pandas as pd
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm

# Project paths
PROJECT_ROOT = Path("/Users/robertjames/Documents/llm_summarization")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DEID_DIR = PROJECT_ROOT / "data" / "deidentified"
DEID_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = PROJECT_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1.1 Configuration: Redaction Rules & Settings

In [ ]:
@dataclass
class RedactionRule:
    name: str
    pattern: str
    flags: int = re.IGNORECASE

DEFAULT_RULES: List[RedactionRule] = [
    RedactionRule("EMAIL", r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b"),
    RedactionRule("URL", r"\bhttps?://\S+\b|\bwww\.\S+\b"),
    RedactionRule("PHONE", r"\b(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b"),
    RedactionRule("SSN", r"\b\d{3}-\d{2}-\d{4}\b"),
    RedactionRule("MRN_LABEL", r"\b(MRN|Medical\s*Record\s*#|Med\s*Rec|Patient\s*ID)\b"),
    RedactionRule("MRN_NUMBER", r"\b\d{6,10}\b"),
    RedactionRule("DATE_MDY", r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"),
    RedactionRule("DATE_TEXT", r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\s+\d{1,2},\s+\d{4}\b"),
    RedactionRule("ADDRESS", r"\b\d{1,6}\s+[A-Z0-9][A-Z0-9\s.-]{2,}\s+(Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Lane|Ln|Drive|Dr)\b"),
    RedactionRule("ZIP", r"\b\d{5}(?:-\d{4})?\b"),
]

CONTEXT_LABELS = {"name", "patient", "dob", "dateofbirth", "birth", "mrn", "acct", "account", "accession"}

@dataclass
class DeidConfig:
    dpi: int = 300
    ocr_lang: str = "eng"
    pad_px: int = 4
    contextual_numeric_redaction: bool = True
    redact_after_label_tokens: int = 6
    enable_broad_numeric_redaction: bool = False

## 1.2 Helper Functions (from deidentify_source_docs.py)

In [ ]:
def pil_to_cv(img: Image.Image) -> np.ndarray:
    arr = np.array(img)
    if arr.ndim == 2:
        return arr
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

def cv_to_pil(arr: np.ndarray) -> Image.Image:
    if arr.ndim == 2:
        return Image.fromarray(arr)
    return Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB))

def compile_rules(rules: List[RedactionRule]) -> List[Tuple[str, re.Pattern]]:
    return [(r.name, re.compile(r.pattern, r.flags)) for r in rules]

def normalize_token(t: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", t.lower())

def ocr_tokens_with_boxes(pil_img: Image.Image, lang: str) -> pd.DataFrame:
    data = pytesseract.image_to_data(pil_img, lang=lang, output_type=pytesseract.Output.DATAFRAME)
    data = data.dropna(subset=["text"])
    data["text"] = data["text"].astype(str)
    data = data[data["text"].str.strip().ne("")]
    data["conf"] = pd.to_numeric(data["conf"], errors="coerce").fillna(-1)
    return data

def token_matches_any_rule(token: str, compiled_rules):
    for name, pat in compiled_rules:
        if pat.search(token):
            return name
    return None

def get_redaction_boxes(ocr_df, compiled_rules, config):
    redactions = []
    ocr_df = ocr_df.copy()
    ocr_df["norm"] = ocr_df["text"].map(normalize_token)
    for idx, row in ocr_df.iterrows():
        token = row["text"]
        rule = token_matches_any_rule(token, compiled_rules)
        if rule == "MRN_NUMBER" and not config.enable_broad_numeric_redaction:
            rule = None
        if rule:
            redactions.append({"rule": rule, "text": token, "left": int(row["left"]),
                               "top": int(row["top"]), "width": int(row["width"]), "height": int(row["height"])})
    if config.contextual_numeric_redaction:
        for _, line_df in ocr_df.groupby(["block_num", "par_num", "line_num"]):
            line_df = line_df.sort_values("word_num")
            norms = line_df["norm"].tolist()
            rows = line_df.to_dict("records")
            for i, n in enumerate(norms):
                if n in CONTEXT_LABELS:
                    for j in range(i + 1, min(i + 1 + config.redact_after_label_tokens, len(rows))):
                        rr = rows[j]
                        if normalize_token(rr["text"]) == "":
                            continue
                        redactions.append({"rule": "CONTEXT_AFTER_LABEL", "text": rr["text"],
                                           "left": int(rr["left"]), "top": int(rr["top"]),
                                           "width": int(rr["width"]), "height": int(rr["height"])})
    seen = set()
    uniq = []
    for r in redactions:
        k = (r["left"], r["top"], r["width"], r["height"], r["text"], r["rule"])
        if k not in seen:
            seen.add(k)
            uniq.append(r)
    return uniq

def apply_redactions_to_image(cv_img, redactions, pad_px):
    out = cv_img.copy()
    h, w = out.shape[:2]
    for r in redactions:
        x1, y1 = max(0, r["left"] - pad_px), max(0, r["top"] - pad_px)
        x2, y2 = min(w, r["left"] + r["width"] + pad_px), min(h, r["top"] + r["height"] + pad_px)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 0), thickness=-1)
    return out

def render_pdf_page_to_pil(doc, page_index, dpi):
    page = doc.load_page(page_index)
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

def images_to_pdf(pil_images, out_path):
    if not pil_images:
        raise ValueError("No pages to write.")
    rgb_imgs = [im.convert("RGB") for im in pil_images]
    rgb_imgs[0].save(out_path, save_all=True, append_images=rgb_imgs[1:])

## 1.3 Patient Mapping Table

Generate a deterministic `case_id` from the original filename so deidentified files can be linked back to observations. The mapping CSV is stored separately.

In [ ]:
def generate_case_id(filename: str) -> str:
    """Deterministic case_id from filename using SHA-256 prefix."""
    h = hashlib.sha256(filename.encode()).hexdigest()[:12]
    return f"CASE_{h.upper()}"

def build_patient_mapping(pdf_dir: Path) -> pd.DataFrame:
    """Build case_id <-> original_filename mapping for all PDFs."""
    rows = []
    for pdf_path in sorted(pdf_dir.rglob("*.pdf")):
        rows.append({
            "case_id": generate_case_id(pdf_path.stem),
            "original_filename": pdf_path.name,
            "original_path": str(pdf_path),
        })
    return pd.DataFrame(rows)

# Build and save the mapping
mapping_df = build_patient_mapping(RAW_DIR)
mapping_path = DEID_DIR / "patient_case_id_mapping.csv"
mapping_df.to_csv(mapping_path, index=False)
print(f"Patient mapping: {len(mapping_df)} cases -> {mapping_path}")
mapping_df.head()

## 1.4 Deidentify Source PDFs

In [ ]:
def deidentify_pdf_folder(
    input_dir: str,
    output_dir: str,
    log_csv_path: str,
    mapping_df: pd.DataFrame,
    rules: List[RedactionRule] = DEFAULT_RULES,
    config: DeidConfig = DeidConfig(),
    glob_pattern: str = "*.pdf",
) -> None:
    in_dir = Path(input_dir)
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    compiled_rules = compile_rules(rules)
    log_rows = []
    pdf_paths = sorted(in_dir.rglob(glob_pattern))

    for pdf_path in tqdm(pdf_paths, desc="Deidentifying PDFs"):
        case_id = generate_case_id(pdf_path.stem)
        try:
            doc = fitz.open(pdf_path)
        except Exception as e:
            log_rows.append({"case_id": case_id, "file": str(pdf_path),
                             "status": "ERROR_OPEN", "error": str(e)})
            continue

        redacted_pages = []
        total_redactions = 0
        for p in range(doc.page_count):
            pil_img = render_pdf_page_to_pil(doc, p, config.dpi)
            ocr_df = ocr_tokens_with_boxes(pil_img, config.ocr_lang)
            redactions = get_redaction_boxes(ocr_df, compiled_rules, config)
            total_redactions += len(redactions)
            cv_img = pil_to_cv(pil_img)
            cv_redacted = apply_redactions_to_image(cv_img, redactions, config.pad_px)
            redacted_pages.append(cv_to_pil(cv_redacted))
            log_rows.append({"case_id": case_id, "file": pdf_path.name,
                             "page": p + 1, "status": "OK",
                             "page_redactions": len(redactions),
                             "sample_rules": ",".join(sorted(set(r["rule"] for r in redactions)))[:200]})

        # Save with case_id filename (deidentified)
        out_path = out_dir / f"{case_id}.pdf"
        try:
            images_to_pdf(redacted_pages, out_path)
        except Exception as e:
            log_rows.append({"case_id": case_id, "file": pdf_path.name,
                             "status": "ERROR_WRITE", "error": str(e)})
            doc.close()
            continue

        log_rows.append({"case_id": case_id, "file": pdf_path.name,
                         "status": "DONE", "pages": doc.page_count,
                         "total_redactions": total_redactions,
                         "output_file": str(out_path)})
        doc.close()

    pd.DataFrame(log_rows).to_csv(log_csv_path, index=False)
    print(f"Done. Redacted PDFs in: {out_dir}")
    print(f"Log: {log_csv_path}")

In [ ]:
# Run deidentification on source PDFs
deidentify_pdf_folder(
    input_dir=str(RAW_DIR),
    output_dir=str(DEID_DIR / "pdfs"),
    log_csv_path=str(DEID_DIR / "deid_pdf_log.csv"),
    mapping_df=mapping_df,
    config=DeidConfig(dpi=300, contextual_numeric_redaction=True, redact_after_label_tokens=8),
)

## 1.5 Deidentify Validation Excel

Strip any free-text PHI columns while preserving the coded status columns (source/human/ai) needed for analysis.

In [ ]:
EXCEL_PATH = RAW_DIR / "merged_llm_summary_validation_datasheet_deidentified.xlsx"
df_raw = pd.read_excel(EXCEL_PATH)
print(f"Loaded validation Excel: {df_raw.shape}")

# Identify status columns (these are safe — coded 0/1/2/3/N/A)
status_cols = [c for c in df_raw.columns if "_status_" in c.lower()]
# Identify potential PHI columns to redact (free text)
phi_candidates = [c for c in df_raw.columns if c not in status_cols
                  and df_raw[c].dtype == "object"
                  and c.lower() not in ["case_id", "observation_id"]]

print(f"Status columns (safe): {len(status_cols)}")
print(f"Free-text columns to inspect: {len(phi_candidates)}")
print(f"  -> {phi_candidates[:10]} ...")

# Apply regex redaction to free-text columns
compiled = compile_rules(DEFAULT_RULES)

def redact_cell(val):
    if pd.isna(val) or not isinstance(val, str):
        return val
    out = val
    for name, pat in compiled:
        out = pat.sub(f"[{name}]", out)
    return out

df_deid = df_raw.copy()
for col in phi_candidates:
    df_deid[col] = df_deid[col].apply(redact_cell)

# Save
deid_excel_path = DEID_DIR / "validation_datasheet_deidentified.xlsx"
df_deid.to_excel(deid_excel_path, index=False)
print(f"Saved deidentified Excel: {deid_excel_path}")

## 1.6 Verification & Summary

In [ ]:
# Verify mapping is complete
log_df = pd.read_csv(DEID_DIR / "deid_pdf_log.csv")
done = log_df[log_df["status"] == "DONE"]
print(f"PDFs processed: {len(done)} / {len(mapping_df)}")
print(f"Total redactions applied: {log_df['total_redactions'].dropna().sum():.0f}")
print(f"\nMapping table preview:")
display(mapping_df.head())
print(f"\nRedaction log preview:")
display(log_df.head(10))